In [1]:
import torch
from transformers import RobertaTokenizer,RobertaModel,RobertaForMaskedLM, pipeline, BertForMaskedLM, BertTokenizer, BertModel, AlbertConfig, AlbertModel, AlbertTokenizer, AlbertForMaskedLM , TFAutoModel, AutoModelForMaskedLM, AutoTokenizer, AutoModelForCausalLM
import math
import numpy as np
import json
import random
import pandas as pd
import tensorflow as tf
from torch.nn import CrossEntropyLoss
from tqdm.notebook import tqdm
import re
from scipy.optimize import linear_sum_assignment

In [ ]:
import logging
import os
import sys
logging.basicConfig(level=logging.INFO)
import warnings
warnings.filterwarnings('ignore')

In [2]:
#Data
def prep_data(source, words):
    source = "test/"+source+".txt"
    file = open(source, "r", encoding = 'utf-8')
    lines = file.readlines()

    data = [[],[],[],[]]
    data_3line = [[],[],[],[]]
    data_3line_unfilled = [[],[],[],[]]
    for n in range(4):
        for l in range(12):
            line = lines[(12*n) + l]
            if line != "\n":
                data[n].append(line)

                prev = lines[l-1].replace("{}", words[n][l-1]) if l > 0 else ""
                next = lines[l+1].replace("{}", words[n][l+1]) if l < 11 else ""

                data_3line[n].append(prev + line + next)

                prev = lines[l-1].replace("{}", '_') if l > 0 else ""
                next = lines[l+1].replace("{}", '_') if l < 11 else ""

                data_3line_unfilled[n].append(prev + line + next)
        
    return [data, data_3line, data_3line_unfilled]

def prep_words(source):
    source = "test/"+source+"_Words.txt"
    file = open(source, "r", encoding = 'utf-8')
    lines = file.readlines()
    
    words = [[],[],[],[]]
    for n in range(4):
        for l in range(12):
            line = lines[(n*12)+l]
            words[n].append(line)
            
    return words

In [17]:
def see_tokens(model, tokenizer, data):
    r = 0
    for d in data:
        for s in d:
            for l in s:
                r += len(tokenizer.tokenize(l))
    return r

In [4]:
def uni_predict(text, model, tokenizer):
    # Tokenized input
    # text = "[CLS] I got restricted because Tom reported my reply [SEP]"
    text = text
    tokenized_text = tokenizer.tokenize(text)
    sentence_score = 0
    indexed_tokens = tokenizer.convert_tokens_to_ids(tokenized_text)
    length = len(tokenized_text)
    tokens_tensor = torch.tensor([indexed_tokens])
    tokens_tensor = tokens_tensor.to('cuda')
    #masked_tensor = torch.tensor([masked_index])
    with torch.no_grad():
        outputs = model(tokens_tensor, labels= tokens_tensor)
    loss = outputs[0]
    sentence_score = -loss
    return sentence_score

In [5]:
def greedy_select(df):
    selected_positions = []
    remaining_rows = set(df.index)
    remaining_columns = set(df.columns)

    while len(remaining_rows) > 0 and len(remaining_columns) > 0:
        min_value = float('inf')
        min_position = None

        # Find the smallest value and its position
        for row in remaining_rows:
            for column in remaining_columns:
                value = df.at[row, column]
                if value < min_value:
                    min_value = value
                    min_position = (row, column)

        # Remove the row and column
        remaining_rows.remove(min_position[0])
        remaining_columns.remove(min_position[1])

        # Add the position to the selected list
        selected_positions.append(min_position)

    return sorted(selected_positions, key=lambda x: x[0])

In [6]:
def score_model(model, tokenizer, data, opts):
    r_scores = []
    df = pd.DataFrame()
    t1_score = 0
    t3_score = 0
    for d in tqdm(data):
        correct = opts[data.index(d)]
        #print(d)
        #print("Correct option is: ", correct)
        scores = {}
        for o in opts:
            sentence = d.replace("{}", o)
            scores.update({o : float(uni_predict(sentence, model, tokenizer).item())})
        df = df.append(scores, ignore_index=True)
        scores = sorted(scores.items(), key=lambda x: x[1], reverse = True)
        i = 0
        for key, value in scores:
            #print(key, ':', value)
            t1_score += ((i == 0) and (key == correct))
            t3_score += ((i < 3) and (key == correct))
            i += 1
        #print()
    
    print("Top 1 Score:", t1_score/12)
    print("Top 3 Score:", t3_score/12)
    r_scores.append(t1_score/12)
    r_scores.append(t3_score/12)
    
    #normalize each row
    df = df.apply(lambda row: row / row.mean(), axis=1)
    
    #Hungarian Algorithm for linear sum assignment optimizes score over all selections
    x,y = linear_sum_assignment(df)
    out = pd.DataFrame({'Word': df.columns[y], 'Sentence': df.index[x]})
    final_score = 0
    for n in range(12):
        final_score += (opts[n] == out.iloc[n]['Word'])
    final_score = final_score/12
    print("Forced Choiced Combinatorially Optimized Score:",final_score)
    r_scores.append(final_score)
    
    #Greedy Selection tries to maximize high confidence picks instead of overall score
    greedy_values = greedy_select(df)
    final_score = 0
    for n in range(12):
        final_score += (opts[n] == greedy_values[n][1])
    final_score = final_score/12
    print("Forced Choiced Greedy Algorithm Score:",final_score)
    r_scores.append(final_score)
    return r_scores

In [19]:
def run_tests(model, tokenizer, lang):
    print(lang)
    word_list = prep_words(lang)
    data = prep_data(lang, word_list)
    print("Token Count:", see_tokens(model, tokenizer, data))
    print("1 Line Data:")
    scores_1_line = [0,0,0,0]
    for i in range(4):
        sc = score_model(model, tokenizer, data[0][i], word_list[i])
        scores_1_line[0] += sc[0]
        scores_1_line[1] += sc[1]
        scores_1_line[2] += sc[2]
        scores_1_line[3] += sc[3]
    print("3 Line Data:")
    scores_3_line = [0,0,0,0]
    for i in range(4):
        sc = score_model(model, tokenizer, data[1][i], word_list[i])
        scores_3_line[0] += sc[0]
        scores_3_line[1] += sc[1]
        scores_3_line[2] += sc[2]
        scores_3_line[3] += sc[3]
    print()
    print("3 Line Data (unfilled):")
    scores_3u_line = [0,0,0,0]
    for i in range(4):
        sc = score_model(model, tokenizer, data[2][i], word_list[i])
        scores_3u_line[0] += sc[0]
        scores_3u_line[1] += sc[1]
        scores_3u_line[2] += sc[2]
        scores_3u_line[3] += sc[3]
    print()
    print("Final values for", lang)
    print("1 line of input")
    print("Top 1 Score:", scores_1_line[0]/4)
    print("Top 3 Score:", scores_1_line[1]/4)
    print("Forced Choiced Combinatorially Optimized Score:", scores_1_line[2]/4)
    print("Forced Choiced Greedy Algorithm Score:", scores_1_line[3]/4)
    print("3 lines of input")
    print("Top 1 Score:", scores_3_line[0]/4)
    print("Top 3 Score:", scores_3_line[1]/4)
    print("Forced Choiced Combinatorially Optimized Score:", scores_3_line[2]/4)
    print("Forced Choiced Greedy Algorithm Score:", scores_3_line[3]/4)
    print("3 lines of input (no fill)")
    print("Top 1 Score:", scores_3u_line[0]/4)
    print("Top 3 Score:", scores_3u_line[1]/4)
    print("Forced Choiced Combinatorially Optimized Score:", scores_3u_line[2]/4)
    print("Forced Choiced Greedy Algorithm Score:", scores_3u_line[3]/4)
    print()

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("sdadas/polish-gpt2-small")
model = AutoModelForCausalLM.from_pretrained("sdadas/polish-gpt2-small").cuda()
run_tests(model, tokenizer, "Czech")
run_tests(model, tokenizer, "Transliterated/Czech-Polish")

Czech
Token Count: 15678
1 Line Data:


  0%|          | 0/12 [00:00<?, ?it/s]

C:\Users\Secon\AppData\Local\Temp\ipykernel_12380\395575093.py:14: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df = df.append(scores, ignore_index=True)
C:\Users\Secon\AppData\Local\Temp\ipykernel_12380\395575093.py:14: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df = df.append(scores, ignore_index=True)
C:\Users\Secon\AppData\Local\Temp\ipykernel_12380\395575093.py:14: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df = df.append(scores, ignore_index=True)
C:\Users\Secon\AppData\Local\Temp\ipykernel_12380\395575093.py:14: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df = df.append(scores, ignore_index=True)
C:\Users\Secon\AppData\L

Top 1 Score: 0.0
Top 3 Score: 0.16666666666666666
Forced Choiced Combinatorially Optimized Score: 0.08333333333333333
Forced Choiced Greedy Algorithm Score: 0.0


C:\Users\Secon\AppData\Local\Temp\ipykernel_12380\395575093.py:14: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df = df.append(scores, ignore_index=True)
C:\Users\Secon\AppData\Local\Temp\ipykernel_12380\395575093.py:14: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df = df.append(scores, ignore_index=True)


...

C:\Users\Secon\AppData\Local\Temp\ipykernel_12380\395575093.py:14: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df = df.append(scores, ignore_index=True)
C:\Users\Secon\AppData\Local\Temp\ipykernel_12380\395575093.py:14: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df = df.append(scores, ignore_index=True)
C:\Users\Secon\AppData\Local\Temp\ipykernel_12380\395575093.py:14: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df = df.append(scores, ignore_index=True)
C:\Users\Secon\AppData\Local\Temp\ipykernel_12380\395575093.py:14: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df = df.append(scores, ignore_index=True)
C:\Users\Secon\AppData\L

Top 1 Score: 0.16666666666666666
Top 3 Score: 0.25
Forced Choiced Combinatorially Optimized Score: 0.25
Forced Choiced Greedy Algorithm Score: 0.25

Final values for Czech
1 line of input
Top 1 Score: 0.08333333333333333
Top 3 Score: 0.22916666666666666
Forced Choiced Combinatorially Optimized Score: 0.08333333333333333
Forced Choiced Greedy Algorithm Score: 0.0625
3 lines of input
Top 1 Score: 0.020833333333333332
Top 3 Score: 0.1875
Forced Choiced Combinatorially Optimized Score: 0.0
Forced Choiced Greedy Algorithm Score: 0.020833333333333332
3 lines of input (no fill)
Top 1 Score: 0.10416666666666666
Top 3 Score: 0.2708333333333333
Forced Choiced Combinatorially Optimized Score: 0.1875
Forced Choiced Greedy Algorithm Score: 0.16666666666666666

Transliterated/Czech-Polish
Token Count: 11500
1 Line Data:


C:\Users\Secon\AppData\Local\Temp\ipykernel_12380\395575093.py:14: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df = df.append(scores, ignore_index=True)


  0%|          | 0/12 [00:00<?, ?it/s]

C:\Users\Secon\AppData\Local\Temp\ipykernel_12380\395575093.py:14: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df = df.append(scores, ignore_index=True)
C:\Users\Secon\AppData\Local\Temp\ipykernel_12380\395575093.py:14: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df = df.append(scores, ignore_index=True)
C:\Users\Secon\AppData\Local\Temp\ipykernel_12380\395575093.py:14: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df = df.append(scores, ignore_index=True)
C:\Users\Secon\AppData\Local\Temp\ipykernel_12380\395575093.py:14: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df = df.append(scores, ignore_index=True)
C:\Users\Secon\AppData\L